# Tiền xử lý & Tạo đặc trưng (Preprocessing & Feature Engineering) - Tối ưu cho LightGBM
**Thực hiện:** Nhóm nghiên cứu & mô hình hóa dữ liệu (Lớp Học máy - UTH)

## Giới thiệu
Dựa trên các phân tích khám phá dữ liệu chuyên sâu tại file `01_EDA.ipynb` và việc đối chiếu với file tiền xử lý thô ban đầu (`02_Preprocessing.ipynb`), file này thực hiện toàn bộ quy trình tiền xử lý và kỹ nghệ đặc trưng tối ưu dành riêng cho mô hình **LightGBM**:

### 1. Phân tích các bước tiền xử lý cốt lõi và kiểm chứng giả thuyết:
*   **Chống rò rỉ dữ liệu (Data Leakage Avoidance):** Biến mục tiêu `Sales_Revenue` được tính bằng $Quantity \times Price$. Để tránh việc mô hình "học vẹt" công thức nhân đơn giản này (khiến mô hình đạt độ chính xác 100% trên tập train nhưng vô dụng trong thực tế vì thực tế lúc dự đoán chưa có `quantity` và `price`), ta **bắt buộc phải loại bỏ** hai cột `quantity` và `price` sau khi tính toán xong mục tiêu.
*   **Loại bỏ các dòng cuối dữ liệu (Lọc mốc thời gian < 01/03/2023):** Dữ liệu năm 2023 chỉ thu thập đến ngày 08/03/2023. Khoảng thời gian 8 ngày của tháng 3/2023 là quá ngắn và không đại diện đủ cho chu kỳ tháng. Việc loại bỏ các dòng này (khoảng 970 dòng) giúp loại bỏ nhiễu của chu kỳ tháng 3 chưa hoàn chỉnh.
*   **Loại bỏ thuộc tính định danh vô ích:** Xóa `customer_id` và `invoice_no` vì tần suất xuất hiện lặp lại của khách hàng bằng 0%.

### 2. Kiểm chứng và Áp dụng Time-based Split thay vì Random Split:
*   *Giả thuyết:* Dữ liệu mua sắm thực tế luôn có yếu tố thời gian và xu hướng thay đổi theo thời gian thực (Time-based). Phân chia ngẫu nhiên (Random Split) dễ làm mô hình sử dụng dữ liệu tương lai để dự đoán quá khứ, không mô phỏng đúng thế giới thực.
*   *Giải pháp:* Thực hiện phân tách dữ liệu theo mốc thời gian **Time-based Split**:
    *   **Tập Train (80.7%):** Toàn bộ giao dịch diễn ra trước ngày **01/10/2022**.
    *   **Tập Test (19.3%):** Toàn bộ giao dịch diễn ra từ ngày **01/10/2022** đến ngày **28/02/2023**.
    *   Cách phân chia này vừa giữ nguyên tỷ lệ 80/20 chuẩn của bài toán, vừa mô phỏng chính xác việc: *"Dùng dữ liệu lịch sử quá khứ để dự đoán tương lai"*.

### 3. Các nâng cấp kỹ thuật đặc trưng (Feature Engineering) cho LightGBM:
*   **Log Transformation:** Biến đổi logarit cho biến mục tiêu `Sales_Revenue` $\rightarrow \log(y + 1)$ để phân phối chuẩn hóa hơn.
*   **Bóc tách thời gian:** Trích xuất các cột đặc trưng `Year`, `Month`, `DayOfWeek` (Thứ trong tuần) và `Is_Weekend` (Cuối tuần).
*   **Phân khúc tuổi tác (`Age_Group`):** Nhóm các độ tuổi thành thế hệ xã hội học (Gen Z, Millennials, Gen X, Boomers).
*   **Ép kiểu dữ liệu danh mục gốc (Native Category Dtype):** Ép kiểu các cột chữ sang kiểu dữ liệu `category` của Pandas để LightGBM sử dụng thuật toán Fisher tối ưu hóa trực tiếp.

## 1. Khai báo Thư viện

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

import warnings
warnings.filterwarnings('ignore')

## 2. Nạp dữ liệu thô và Áp dụng bộ lọc thời gian

In [2]:
# Nạp dữ liệu từ thư mục raw-data bên ngoài folder LightGBM_vqd
data_path = "../data/raw-data/customer_shopping_data.csv"
if not os.path.exists(data_path):
    data_path = "./practice_2/data/raw-data/customer_shopping_data.csv"

df = pd.read_csv(data_path)
print(f"Kích thước dữ liệu gốc ban đầu: {df.shape}")

# Chuyển invoice_date sang định dạng datetime
df['invoice_date'] = pd.to_datetime(df['invoice_date'], format='%d/%m/%Y')

# Kiểm chứng giả thuyết lọc mốc thời gian: Loại bỏ các giao dịch thuộc tháng 3/2023 để tránh chu kỳ dở dang
df_filtered = df[df['invoice_date'] < '2023-03-01'].copy()
print(f"Kích thước dữ liệu sau khi lọc (< 01/03/2023): {df_filtered.shape}")
print(f"Số dòng bị loại bỏ: {len(df) - len(df_filtered):,}")

Kích thước dữ liệu gốc ban đầu: (99457, 10)
Kích thước dữ liệu sau khi lọc (< 01/03/2023): (98487, 10)
Số dòng bị loại bỏ: 970


## 3. Tạo biến mục tiêu và Thực hiện Log Transformation
Chúng ta tạo biến mục tiêu `Sales_Revenue` và áp dụng phép biến đổi logarit $\log(y + 1)$ để ổn định phương sai cho LightGBM.

In [3]:
# Tính toán doanh thu thực tế
df_filtered['Sales_Revenue'] = df_filtered['quantity'] * df_filtered['price']

# Áp dụng Log Transformation cho biến mục tiêu
df_filtered['Sales_Revenue_Log'] = np.log1p(df_filtered['Sales_Revenue'])

print("Thống kê mô tả doanh thu sau khi biến đổi Log:")
print(df_filtered['Sales_Revenue_Log'].describe())

Thống kê mô tả doanh thu sau khi biến đổi Log:
count    98487.000000
mean         6.334213
std          2.031890
min          1.829376
25%          4.922532
50%          6.398878
75%          7.901644
max         10.175459
Name: Sales_Revenue_Log, dtype: float64


## 4. Kỹ nghệ Đặc trưng (Feature Engineering)
Chúng ta tiến hành tạo các đặc trưng thời gian và phân khúc độ tuổi thế hệ.

In [4]:
# A. Trích xuất đặc trưng thời gian
df_filtered['Year'] = df_filtered['invoice_date'].dt.year.astype(str)
df_filtered['Month'] = df_filtered['invoice_date'].dt.month.astype(str)
df_filtered['DayOfWeek'] = df_filtered['invoice_date'].dt.day_name()
df_filtered['Is_Weekend'] = df_filtered['DayOfWeek'].isin(['Saturday', 'Sunday']).astype(int)

# B. Trích xuất nhóm tuổi thế hệ (Age Binning)
def get_age_group(age):
    if age <= 25:
        return 'Gen_Z'
    elif age <= 41:
        return 'Millennials'
    elif age <= 57:
        return 'Gen_X'
    else:
        return 'Boomers'

df_filtered['Age_Group'] = df_filtered['age'].apply(get_age_group)

# Hiển thị tỷ lệ phân bổ của đặc trưng nhóm tuổi mới
print("Tỷ lệ phân phối nhóm tuổi thế hệ:")
print(df_filtered['Age_Group'].value_counts(normalize=True) * 100)

Tỷ lệ phân phối nhóm tuổi thế hệ:
Age_Group
Millennials    31.003077
Gen_X          30.656838
Boomers        22.899469
Gen_Z          15.440617
Name: proportion, dtype: float64


## 5. Chọn lọc đặc trưng và Loại bỏ các cột chống rò rỉ dữ liệu
Chúng ta loại bỏ các cột định danh ngẫu nhiên và các cột cấu thành nên doanh thu trực tiếp (`quantity` và `price`) để tránh overfitting và rò rỉ dữ liệu.

In [5]:
# Danh sách các cột đặc trưng độc lập dùng để huấn luyện mô hình
features_to_keep = [
    'gender', 'age', 'Age_Group', 'category', 'payment_method', 
    'shopping_mall', 'Year', 'Month', 'DayOfWeek', 'Is_Weekend'
]

# Khởi tạo X và y kèm thông tin ngày tháng để phân chia tập dữ liệu
X = df_filtered[features_to_keep].copy()
X['invoice_date'] = df_filtered['invoice_date'] # Giữ lại tạm thời để chia dữ liệu
y = df_filtered['Sales_Revenue_Log'].copy() # Sử dụng doanh thu đã biến đổi logarit
y_original = df_filtered['Sales_Revenue'].copy() # Lưu trữ doanh thu gốc để đối chiếu khi đánh giá

print(f"Cấu trúc ma trận đặc trưng X: {X.shape}")

Cấu trúc ma trận đặc trưng X: (98487, 11)


## 6. Ép kiểu Categorical gốc (Native Category Dtype)
Đây là kỹ thuật quan trọng nhất giúp LightGBM tự động tìm điểm chia nhánh tối ưu bằng thuật toán Fisher thay vì One-Hot.

In [6]:
categorical_cols = ['gender', 'Age_Group', 'category', 'payment_method', 'shopping_mall', 'Year', 'Month', 'DayOfWeek']

for col in categorical_cols:
    X[col] = X[col].astype('category')

print("Thông tin kiểu dữ liệu của các cột trong X:")
print(X.dtypes)

Thông tin kiểu dữ liệu của các cột trong X:
gender                  category
age                        int64
Age_Group               category
category                category
payment_method          category
shopping_mall           category
Year                    category
Month                   category
DayOfWeek               category
Is_Weekend                 int64
invoice_date      datetime64[ns]
dtype: object


## 7. Phân chia tập Train/Test theo thời gian (Time-based Split)

Để đảm bảo mô phỏng đúng thực tế "Dùng dữ liệu quá khứ dự đoán tương lai", chúng ta lấy mốc phân cách là ngày **01/10/2022**:
*   **Tập Train (Giao dịch trước 01/10/2022):** ~80.7% lượng dữ liệu.
*   **Tập Test (Giao dịch từ 01/10/2022 trở đi):** ~19.3% lượng dữ liệu.

In [7]:
split_date = '2022-10-01'

# Tạo mặt nạ lọc dòng
train_mask = X['invoice_date'] < split_date
test_mask = X['invoice_date'] >= split_date

# Phân chia các tập dữ liệu
X_train = X[train_mask].drop(columns=['invoice_date'])
X_test = X[test_mask].drop(columns=['invoice_date'])

y_train = y[train_mask]
y_test = y[test_mask]
y_test_orig = y_original[test_mask]

print(f"Tổng số dòng: {len(X):,}")
print(f"Kích thước tập X_train (Trước {split_date}): {X_train.shape[0]:,} ({X_train.shape[0]/len(X)*100:.2f}%)")
print(f"Kích thước tập X_test  (Từ {split_date} trở đi):  {X_test.shape[0]:,} ({X_test.shape[0]/len(X)*100:.2f}%)")

Tổng số dòng: 98,487
Kích thước tập X_train (Trước 2022-10-01): 79,521 (80.74%)
Kích thước tập X_test  (Từ 2022-10-01 trở đi):  18,966 (19.26%)


## 8. Lưu trữ Dữ liệu chuẩn bị huấn luyện

In [8]:
# Tạo thư mục con data_LightGBM/ready_for_train bên trong thư mục làm việc hiện tại
output_dir = "./data_LightGBM/ready_for_train/"
os.makedirs(output_dir, exist_ok=True)

# Dump dữ liệu ra các file pkl để phục vụ bước train mô hình tiếp theo
joblib.dump(X_train, f"{output_dir}X_train.pkl")
joblib.dump(X_test, f"{output_dir}X_test.pkl")
joblib.dump(y_train, f"{output_dir}y_train.pkl")
joblib.dump(y_test, f"{output_dir}y_test.pkl")
joblib.dump(y_test_orig, f"{output_dir}y_test_original.pkl") 

print(f"\n=> HOÀN THÀNH: Đã lưu trữ toàn bộ dữ liệu tiền xử lý theo Time-based Split tại: {output_dir}")


=> HOÀN THÀNH: Đã lưu trữ toàn bộ dữ liệu tiền xử lý theo Time-based Split tại: ./data_LightGBM/ready_for_train/
